# Telecom-T2C-Trainer

Continue-LoRA (QLoRA) fine-tuning of `google/gemma-4-12B-it` on the telecom NL -> TIR (PASS_0-4) multi-turn dataset.

**This notebook only orchestrates** — all logic lives in `src/`. The only file you should ever need to edit between runs is `configs/experiment.yaml`.

Run cells top to bottom. Sections: Runtime Check, Install, Configuration, Load Dataset, Dataset Statistics, Token Analysis, Load Model, Load Adapter, Train, Evaluate, Save, Smoke Test.

## 1. Runtime Check

In [ ]:
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Locate the Telecom-T2C project root from any Colab/local starting cwd."""
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for child in start.glob("*/"):
        if (child / "src").is_dir() and (child / "configs").is_dir():
            return child
    raise RuntimeError(
        "Could not locate the Telecom-T2C project root (a directory containing both "
        "'src/' and 'configs/'). If running in Colab, cd into the cloned/uploaded repo "
        "directory first, e.g.:\n  %cd /content/Telecom-T2C"
    )


PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src import utils

logger = utils.setup_logging()
gpu = utils.detect_gpu()
print(f"GPU: {gpu.name} | family={gpu.family} | vram={gpu.vram_gb:.1f} GB | bf16={gpu.bf16_supported}")

import torch

print(f"torch: {torch.__version__} | built for CUDA {torch.version.cuda}")
print(
    "Note: requirements.txt deliberately never pins/reinstalls torch — Colab's "
    "pre-installed build is already matched to its own CUDA driver and to "
    "torchaudio/torchvision. If a later `pip install` step ever reports a "
    "different torch version here after a restart, see README Troubleshooting "
    "('PyTorch and TorchAudio were compiled with different CUDA versions')."
)

if gpu.family == "CPU":
    raise RuntimeError(
        "No GPU detected. In Colab: Runtime -> Change runtime type -> select a GPU "
        "(A100 recommended for this 12B model)."
    )
elif gpu.family != "A100":
    print(
        f"Warning: this notebook's defaults are tuned for A100. Detected {gpu.family} — "
        "Section 7 (Load Model) will print recommended overrides for configs/experiment.yaml."
    )

## 2. Install

Re-run this cell after any runtime restart, and again any time you pull a `requirements.txt` or notebook change — it always runs the installs with `--upgrade`, so a re-run actually applies updated floor pins instead of silently no-op'ing because an old version already "satisfies" a loosened constraint.

**Important: "Runtime -> Restart session" only restarts the Python process — it does NOT reset installed packages.** Every `pip install`/`--upgrade` you've ever run in this VM is still on disk. If you've already tried "re-run Install, then Restart session" a couple of times and keep hitting *different* missing-symbol errors each time (not the same one repeating), that's a sign of accumulated package drift, not a new bug — go to **Runtime -> Disconnect and delete runtime**, reconnect (a genuinely fresh VM), and run the Install cell once from a clean slate before troubleshooting further.

**This cell installs in four phases, not one flat `pip install -r requirements.txt`** — mirroring [Unsloth's own official Colab recipe for newer Gemma 4 variants](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(26B_A4B)-Vision.ipynb), adapted for `google/gemma-4-12B-it` specifically. See `requirements.txt`'s comments for the full reasoning: in short, `unsloth`/`unsloth_zoo`'s own PyPI metadata caps `transformers<=5.5.0`, but this exact model's `gemma4_unified` architecture needs `transformers>=5.10.0` — resolving both in one plain pip call forces pip to backtrack `unsloth`/`unsloth_zoo` down to an ancient, pre-Gemma-4 release instead. Phases 2 and 3 use `--no-deps` specifically to prevent that.

**Known Colab quirks** (both self-diagnosed by the version-check cell below):
- `numpy.dtype size changed, may indicate binary incompatibility` — pip reconciled numpy/pandas/pyarrow versions inside this already-running kernel (the old binaries were compiled against Colab's pre-installed numpy).
- `Could not import module 'X'` from `peft` or `trl` — the installed peft/trl release predates the transformers version this notebook installs; phase 2 below floor-pins both specifically to stay ahead of this.

Either way: **re-run this Install section's pip-install cell**, then **Runtime -> Restart session**, then re-run the notebook from Section 1.

In [ ]:
import re
import subprocess
import sys

import torch

from src import utils

# Phase 1: everything that resolves normally (no --no-deps needed) —
# huggingface_hub, datasets, sentencepiece, PyYAML, protobuf, hf_transfer,
# safetensors, wandb, nvidia-ml-py, pytest. --upgrade is required, not
# optional: without it, pip leaves an already-installed package alone as
# long as it satisfies a floor constraint (e.g. peft==0.14.0 already
# "satisfies" peft>=0.14.0), so a re-run after pulling an updated
# requirements.txt would otherwise silently do nothing.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "-r", str(PROJECT_ROOT / "requirements.txt"),
    ]
)

# Phase 2: the correlated Unsloth ML stack, installed together with
# --no-deps so pip never attempts to resolve unsloth_zoo's declared
# transformers<=5.5.0 ceiling against this project's actual transformers
# version (installed separately in Phase 3) — see requirements.txt's top
# comment for the full, empirically-confirmed reasoning. Floors on
# bitsandbytes/accelerate/peft/trl match this project's own
# previously-validated versions (--no-deps skips dependency RESOLUTION,
# not version constraints on the packages named directly in this command).
# xformers' exact pin is chosen dynamically from the installed torch
# version, copied verbatim from Unsloth's own official Colab recipe (see
# notebook Section 2 markdown for the link) since picking the wrong one
# for a given torch release is a real, separate compatibility trap.
torch_version = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
xformers_pin = "xformers==" + {
    "2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2",
}.get(torch_version, "0.0.34")
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "unsloth_zoo", "bitsandbytes>=0.46.1,!=0.48.0", "accelerate>=1.8",
        xformers_pin, "peft>=0.19.1", "trl>=0.15.0", "triton", "unsloth",
    ]
)

# Phase 3: torchao, also --no-deps and kept separate from Phase 2 (matching
# the official recipe) — unsloth_zoo declares torchao>=0.13.0 as a genuine
# dependency of its own (not just something transformers' quantizer
# machinery incidentally imports), so this project installs a real,
# specific floor for it rather than uninstalling it outright. An earlier
# version of this cell uninstalled torchao entirely after hitting a
# present-but-broken torchao/torch op mismatch
# (`torch.ops._c10d_functional._wrap_tensor_autograd`); this floor mirrors
# what Unsloth's own official notebook installs for a newer Gemma 4
# variant, on the theory that an unconstrained `pip install torchao`
# (letting pip pick whatever's latest) was the actual cause of that
# mismatch, not torchao categorically. utils.disable_unused_transformers_backends()
# below still neutralizes transformers' OWN torchao-mediated quantizer
# import path regardless (this project never uses TorchAO quantization
# directly), as a second line of defense in case this floor still doesn't
# match a given Colab image's torch build.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "torchao>=0.16.0",
    ]
)

# Phase 4: transformers + tokenizers, --no-deps, installed LAST and
# separately from Phase 2 — this is what actually gets this project past
# unsloth_zoo's declared transformers<=5.5.0 ceiling. transformers==5.10.2
# is the narrow window that both recognizes google/gemma-4-12B-it's
# `gemma4_unified` architecture (registered starting at transformers
# 5.10.0) and stays below the version where unsloth's own exec()-based
# patching (unsloth/models/_utils.py) is confirmed broken (5.12.1 raised
# `NameError: name 'auto_docstring' is not defined`) — see
# requirements.txt's top comment for the full history. tokenizers' range
# matches transformers==5.10.2's own declared requirement exactly.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "transformers==5.10.2", "tokenizers>=0.22.0,<=0.23.0",
    ]
)

# torchaudio is not needed anywhere in this text-only project, and has been
# confirmed — in a separate incident from the torchao one above — to ship
# on some Colab images with an internal circular-import bug in its own
# CUDA-version check, which crashes unrelated imports (nearly any Auto*
# class transitively pulls it in via transformers' generic audio-loss
# module). Uninstalling it entirely is the fix: transformers' own
# optional-dependency detection correctly skips torchaudio-dependent code
# paths when the package isn't present at all, rather than attempting to
# use a present-but-broken one. check=False because it's a no-op (not an
# error) if it isn't installed in the first place.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"],
    check=False,
)

# Belt-and-suspenders on top of Phase 3 above: transformers' own
# is_torchao_available() check only confirms torchao is *present*, not that
# importing it actually succeeds — a real, previously-confirmed failure
# mode on Colab images where the installed torchao build doesn't match the
# installed torch build. This project never uses TorchAO quantization
# directly (always bitsandbytes 4-bit via Unsloth), so neutralizing this
# check entirely is safe regardless of whether Phase 3's floor actually
# works on a given image. Called as early as possible (before Section 2's
# own diagnostic `import unsloth` below, and well before Sections 4/7's
# transformers imports) since the check is `@lru_cache`d — see
# utils.disable_unused_transformers_backends()'s docstring for the full
# reasoning.
utils.disable_unused_transformers_backends()

In [ ]:
import importlib
import subprocess
import sys
import traceback

_RESTART_HINT = (
    "Re-run this Install section's pip-install cell, then Runtime -> Restart "
    "session, then re-run the notebook from Section 1. Package versions were "
    "reconciled inside this already-running kernel and it needs a fresh process. "
    "If that doesn't help (or you've already tried it and keep seeing DIFFERENT "
    "missing-symbol errors each time), your disk has accumulated package drift "
    "across repeated installs — Restart session does not reset it. Use "
    "Runtime -> Disconnect and delete runtime for a genuinely clean VM instead."
)


def _installed_version(pkg: str) -> str:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", pkg], capture_output=True, text=True
    )
    for line in result.stdout.splitlines():
        if line.startswith("Version:"):
            return line.removeprefix("Version:").strip()
    return "not installed"


# Note: peft is NOT checked directly here even though it's a real dependency —
# a standalone `import peft` can fail (a known, separate upstream compatibility
# gap — see README Troubleshooting) without necessarily blocking the actual
# Unsloth-mediated LoRA path this project uses, so checking it here could show
# a scary FAILED that doesn't reflect whether the notebook will actually work.
failures = []
for pkg in ("transformers", "datasets", "unsloth", "unsloth_zoo", "trl", "bitsandbytes", "accelerate"):
    try:
        module = importlib.import_module(pkg)
        print(f"{pkg}: {getattr(module, '__version__', 'ok')}")
    except Exception:
        print(f"{pkg}: FAILED to import (pip-installed version: {_installed_version(pkg)})")
        traceback.print_exc()  # full chained traceback — the real root cause, not just the outer wrapper
        print()
        failures.append(pkg)

if failures:
    print(f"{len(failures)} package(s) failed to import: {', '.join(failures)}")
    print(_RESTART_HINT)
    print(
        "If the traceback above doesn't make the fix obvious, copy the FULL output "
        "(including the traceback, not just the last line) when asking for help."
    )

## 3. Configuration

Loads `configs/experiment.yaml` — edit that file, not this cell, to change settings.

In [ ]:
from src import config as config_mod
from src import manifest as manifest_mod

CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment.yaml"
experiment_config = config_mod.load_config(CONFIG_PATH)

for warning in config_mod.validate_config(experiment_config):
    print(f"WARNING: {warning}")

prior_manifest = manifest_mod.load_prior_manifest(experiment_config.model.continue_adapter)
experiment_config = manifest_mod.apply_prior_manifest_inheritance(experiment_config, prior_manifest)

utils.set_seed(experiment_config.identity.seed)

if experiment_config.drive.google_drive_directory:
    utils.mount_google_drive(experiment_config.drive.drive_mount_point)

run_dir = config_mod.resolve_run_dir(experiment_config)
config_mod.save_resolved_config(experiment_config, run_dir / "config.yaml")
print(f"Run directory: {run_dir}")
print(f"Base model: {experiment_config.model.base_model}")
print(f"Continue adapter: {experiment_config.model.continue_adapter or '(none — fresh LoRA init)'}")

## 4. Load Dataset

In [ ]:
from src import tokenizer as tokenizer_mod
from src import dataset as dataset_mod

hf_token = tokenizer_mod.resolve_hf_token(experiment_config.model.hf_token_env_var)
tokenizer = tokenizer_mod.load_tokenizer(experiment_config.model.base_model, hf_token)

loader = dataset_mod.DatasetLoader(experiment_config.data, tokenizer, experiment_config.identity.seed)
train_ds = loader.load_train()
val_ds = loader.load_validation()
golden_ds = loader.load_golden()

print(
    f"train={len(train_ds):,}  "
    f"val={len(val_ds):,}" if val_ds is not None else "val=0",
    f"golden={len(golden_ds):,}" if golden_ds is not None else "golden=0 (skipped)",
)

## 5. Dataset Statistics

Tokenizes each split once; the results are reused by Section 6 (Token Analysis) rather than re-tokenizing — the train split alone is ~91k conversations.

In [ ]:
from src import statistics as statistics_mod
from src import model as model_mod

gpu_profile = model_mod.detect_gpu_profile(experiment_config.hardware.gpu_profile_override)
if gpu_profile.family != "A100":
    print(f"GPU profile note: {gpu_profile.notes}")
    print(
        f"  recommended batch_size={gpu_profile.recommended_batch_size}, "
        f"gradient_accumulation={gpu_profile.recommended_gradient_accumulation}, "
        f"max_train_samples={gpu_profile.recommended_max_train_samples}"
    )

token_analyses = {}
for split_name, split_ds in (("train", train_ds), ("val", val_ds), ("golden", golden_ds)):
    if split_ds is None:
        continue
    stats = statistics_mod.compute_dataset_statistics(
        split_ds, tokenizer, experiment_config.data.max_seq_length, split_name,
        experiment_config.training.batch_size, experiment_config.training.gradient_accumulation,
        experiment_config.training.epochs, experiment_config.training.packing, gpu_profile.family,
    )
    statistics_mod.print_statistics_report(stats)
    token_analyses[split_name] = statistics_mod.analyze_tokens(
        split_ds, tokenizer, experiment_config.data.max_seq_length
    )
    statistics_mod.display_histogram(
        token_analyses[split_name].per_example_lengths,
        title=f"\nToken-length histogram: {split_name}",
    )

## 6. Token Analysis

Reuses the tokenization already computed in Section 5.

In [ ]:
for split_name, analysis in token_analyses.items():
    print(f"=== Token analysis: {split_name} ===")
    print(f"  mean:   {analysis.mean:.1f}")
    print(f"  median: {analysis.median:.1f}")
    print(f"  p95:    {analysis.p95:.1f}")
    print(f"  max:    {analysis.max}")
    print(
        f"  exceeding max_seq_length ({experiment_config.data.max_seq_length}): "
        f"{analysis.num_exceeding_max_seq_length}"
    )

## 7. Load Model

Loads the base model via Unsloth's `FastModel` — model and tokenizer come back together (Unsloth configures both in lockstep), so the next cell reassigns `tokenizer` to the returned value, since everything from here on (training, generation) must use that tokenizer, not the one loaded in Section 4 for dataset statistics.

**Not validated on real hardware by this project (no GPU available during development).** If this cell fails, see README "Model backend" and Troubleshooting for known unsloth/transformers compatibility issues and how to work around them (re-running Install to pick up a fix is usually the first thing to try).

In [ ]:
base_model, tokenizer = model_mod.load_base_model(
    experiment_config.model, experiment_config.data.max_seq_length, hf_token
)

## 8. Load Adapter

Continues `model.continue_adapter` if set in the config, otherwise initializes a fresh LoRA adapter. Never merges the adapter into the base model.

In [ ]:
peft_model = model_mod.attach_lora(
    base_model, experiment_config.lora, experiment_config.model.continue_adapter,
    experiment_config.training.gradient_checkpointing,
)

## 9. Train

Long-running cell. Automatically resumes from the latest checkpoint under this run's `adapter/` directory if `reproducibility.resume_training` is true and a prior interrupted run is found.

In [ ]:
from src import checkpoint as checkpoint_mod
from src import wandb_logger as wandb_logger_mod
from src import trainer as trainer_mod

run_metadata = {
    "experiment_name": experiment_config.identity.experiment_name,
    "dataset_version": experiment_config.identity.dataset_version,
    "lora_version": experiment_config.identity.lora_version,
    "generator_version": experiment_config.identity.generator_version,
    "validator_version": experiment_config.identity.validator_version,
    "git_hash": utils.get_git_hash(),
    # Hyperparameters — wandb's recommended use of config= is to pass
    # exactly these, so runs are comparable side-by-side in the UI.
    "base_model": experiment_config.model.base_model,
    "epochs": experiment_config.training.epochs,
    "learning_rate": experiment_config.training.learning_rate,
    "batch_size": experiment_config.training.batch_size,
    "gradient_accumulation": experiment_config.training.gradient_accumulation,
    "packing": experiment_config.training.packing,
    "max_seq_length": experiment_config.data.max_seq_length,
    "lora_r": experiment_config.lora.lora_r,
    "lora_alpha": experiment_config.lora.lora_alpha,
    "lora_dropout": experiment_config.lora.lora_dropout,
    "train_rows": len(train_ds),
}

wb_logger = wandb_logger_mod.WandbLogger(experiment_config.wandb, run_metadata)
wb_logger.init(job_type="train")

eval_dataset_for_training = (
    golden_ds if experiment_config.data.eval_source == "golden" and golden_ds is not None else val_ds
)

sft_trainer = trainer_mod.train(
    experiment_config, peft_model, tokenizer, train_ds, eval_dataset_for_training,
    run_dir, wb_logger, run_metadata,
)
trainer_mod.save_best_model(sft_trainer, run_dir / "adapter", tokenizer)

## 10. Evaluate

Loss-based validation metrics, plus a golden generation-eval + benchmark report if `data.golden_path` is configured (skipped gracefully otherwise).

In [ ]:
from src import evaluator as evaluator_mod
from src import benchmark as benchmark_mod
from src import inference as inference_mod

val_metrics = None
if eval_dataset_for_training is not None:
    val_metrics = evaluator_mod.evaluate_validation(sft_trainer)
    print("Validation metrics:", val_metrics)
    wb_logger.set_summary({f"final/{k}": v for k, v in val_metrics.items()})

report = benchmark_mod.run_benchmark(
    experiment_config, run_dir / "adapter", run_dir, golden_ds,
    hf_token=hf_token, val_metrics=val_metrics,
)
benchmark_mod.write_benchmark_report(report, run_dir / "metrics" / "benchmark_report.json")
print("Golden metrics:", report.golden_metrics)
print("PASS metric status (interfaces only, not yet implemented):", report.pass_metric_status)
if report.golden_metrics:
    wb_logger.set_summary({f"final/golden_{k}": v for k, v in report.golden_metrics.items()})

## 11. Save

Writes `manifest.json`, zips the adapter, syncs the run to Google Drive, and uploads artifacts to wandb (each step no-ops gracefully if Drive/wandb aren't available).

In [ ]:
final_manifest = manifest_mod.build_manifest(
    experiment_config, prior_manifest, len(train_ds),
    len(val_ds) if val_ds is not None else 0,
    len(golden_ds) if golden_ds is not None else None,
    run_dir.name,
)
manifest_mod.write_manifest(final_manifest, run_dir / "manifest.json")

zip_path = checkpoint_mod.zip_adapter(
    run_dir / "adapter", run_dir / f"{experiment_config.identity.experiment_name}.zip"
)
checkpoint_mod.sync_run_to_drive(
    run_dir, experiment_config.drive.google_drive_directory, experiment_config.drive.drive_mount_point
)

exp_name = experiment_config.identity.experiment_name
for artifact_path, artifact_type, artifact_name in (
    (run_dir / "manifest.json", "manifest", f"{exp_name}-manifest"),
    (run_dir / "adapter", "model", f"{exp_name}-adapter"),
    (run_dir / "predictions", "predictions", f"{exp_name}-predictions"),
    (run_dir / "metrics", "metrics", f"{exp_name}-metrics"),
    (run_dir / "config.yaml", "config", f"{exp_name}-config"),
):
    wb_logger.upload_artifact(artifact_path, artifact_type, artifact_name)

wb_logger.finish()
print(f"Run saved to {run_dir}")
print(f"Adapter zip: {zip_path}")

## 12. Smoke Test

Reloads the saved adapter fresh (as if after a runtime restart) and generates from one training example's prompt turns, for a quick sanity check against the gold reply.

In [ ]:
inf_model, inf_tokenizer = inference_mod.load_model_for_inference(
    experiment_config.model, experiment_config.data.max_seq_length,
    str(run_dir / "adapter"), hf_token,
)

sample_messages = train_ds[0]["messages"]
first_assistant_idx = next(i for i, m in enumerate(sample_messages) if m["role"] == "assistant")
prompt_messages = sample_messages[:first_assistant_idx]
gold = sample_messages[first_assistant_idx]["content"]

generated = inference_mod.run_smoke_test(
    inf_model, inf_tokenizer, prompt_messages,
    max_new_tokens=experiment_config.evaluation.max_new_tokens_eval,
)
print("\n=== GOLD ===")
print(gold[:1500])